In [1]:
"""
Multi-Layer Perceptron (MLP) - XOR Problem
===========================================
Goal: Learn the XOR logic gate using a neural network

WHAT IS XOR?
  Input A | Input B | Output
  --------|---------|-------
    0     |    0    |   0
    0     |    1    |   1
    1     |    0    |   1
    1     |    1    |   0

WHY IS XOR INTERESTING?
  - A single perceptron (one layer) CANNOT learn XOR.
    XOR is not "linearly separable" — you can't draw one straight line
    to separate the 0s from the 1s.
  - But an MLP with a hidden layer CAN learn it!
    This is the classic proof that hidden layers matter.

FLOW:
  XOR Data → Train MLP → See it learn → Plot decision boundary
"""

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ──────────────────────────────────────────────────────────────
# STEP 1: CREATE THE XOR DATASET
# ──────────────────────────────────────────────────────────────

# The 4 XOR truth table entries
X = np.array([
    [0, 0],   # → 0
    [0, 1],   # → 1
    [1, 0],   # → 1
    [1, 1],   # → 0
])

y = np.array([0, 1, 1, 0])

print("XOR Dataset:")
print("-" * 30)
print("  A  |  B  | Output")
print("-" * 30)
for (a, b), out in zip(X, y):
    print(f"  {a}  |  {b}  |   {out}")
print("-" * 30)

# ──────────────────────────────────────────────────────────────
# STEP 2: THE MLP
# ──────────────────────────────────────────────────────────────

class MLP:
    """
    MLP for XOR — Architecture:
      [2 inputs] → [4 hidden neurons (Tanh)] → [1 output (Sigmoid)]

    Why Tanh in hidden layers?
      Tanh outputs values between -1 and +1 (zero-centered).
      This helps gradients flow better than sigmoid, making XOR
      converge faster and more reliably.

    Why Sigmoid in the output?
      We need a probability between 0 and 1 for binary classification.
    """

    def __init__(self, learning_rate=0.1):
        """
        Initialize weights randomly (small values near 0).
        W1: shape (2, 4) — 2 inputs going into 4 hidden neurons
        W2: shape (4, 1) — 4 hidden neurons going into 1 output
        """
        self.lr = learning_rate

        np.random.seed(3)    # so you get the same result every run
        self.W1 = np.random.randn(2, 4)   # Input → Hidden
        self.b1 = np.zeros((1, 4))

        self.W2 = np.random.randn(4, 1)   # Hidden → Output
        self.b2 = np.zeros((1, 1))

        self.losses = []

    # ── Activation ───────────────────────────────────────────

    def sigmoid(self, z):
        """Squashes values to (0, 1). Used in all layers here."""
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, a):
        """
        Derivative of sigmoid, given the OUTPUT a = sigmoid(z).
        Formula: a * (1 - a)
        Needed during backpropagation.
        """
        return a * (1 - a)

    def tanh(self, z):
        """
        Tanh activation: outputs between -1 and +1.
        Works better than sigmoid for hidden layers in XOR
        because it's zero-centered — gradients flow more cleanly.
        """
        return np.tanh(z)

    def tanh_derivative(self, a):
        """
        Derivative of tanh, given the OUTPUT a = tanh(z).
        Formula: 1 - a²
        """
        return 1 - a ** 2

    # ── Loss ─────────────────────────────────────────────────

    def mse_loss(self, y_true, y_pred):
        """
        Mean Squared Error: average of (prediction - truth)²
        Simple and works well for XOR since we have only 4 samples.
        MSE gives a clean downward loss curve as the model improves.
        """
        return np.mean((y_pred.flatten() - y_true) ** 2)

    # ── Forward Pass ─────────────────────────────────────────

    def forward(self, X):
        """
        Input travels forward through each layer.

          Hidden layer:
            z1 = X @ W1 + b1
            a1 = sigmoid(z1)

          Output layer:
            z2 = a1 @ W2 + b2
            a2 = sigmoid(z2)  ← final prediction (0 to 1)
        """
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.tanh(self.z1)          # tanh for hidden layer

        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.sigmoid(self.z2)

        return self.a2

    # ── Backward Pass ────────────────────────────────────────

    def backward(self, X, y_true):
        """
        Error travels backward. We compute how much each weight
        contributed to the mistake, then nudge it in the right direction.

        Output layer:
          delta2 = (prediction - truth) * sigmoid'(output)

        Hidden layer:
          delta1 = (delta2 @ W2.T) * sigmoid'(hidden_output)

        Then update:
          W = W - lr * gradient
        """
        m = X.shape[0]

        # --- Output layer ---
        # For MSE loss + sigmoid: d(MSE)/d(z2) = (pred - true) * sigmoid'(a2)
        delta2 = (self.a2 - y_true.reshape(-1, 1)) * self.sigmoid_derivative(self.a2)
        dW2 = (self.a1.T @ delta2) / m
        db2 = np.mean(delta2, axis=0, keepdims=True)

        # --- Hidden layer ---
        delta1 = (delta2 @ self.W2.T) * self.tanh_derivative(self.a1)
        dW1 = (X.T @ delta1) / m
        db1 = np.mean(delta1, axis=0, keepdims=True)

        # --- Update weights ---
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    # ── Training Loop ────────────────────────────────────────

    def train(self, X, y, epochs=10000):
        """
        XOR has only 4 samples — we use the full dataset each epoch
        (no mini-batches needed here).
        We train for many epochs because XOR requires the network
        to really fine-tune its weights.
        """
        for epoch in range(1, epochs + 1):
            y_pred = self.forward(X)
            loss = self.mse_loss(y, y_pred)
            self.losses.append(loss)
            self.backward(X, y)

            if epoch % 1000 == 0:
                preds = (y_pred >= 0.5).astype(int).flatten()
                acc = np.mean(preds == y) * 100
                print(f"  Epoch {epoch:5d} | Loss: {loss:.6f} | Accuracy: {acc:.1f}%")

    def predict(self, X):
        probs = self.forward(X)
        return (probs >= 0.5).astype(int).flatten()


# ──────────────────────────────────────────────────────────────
# STEP 3: TRAIN THE MODEL
# ──────────────────────────────────────────────────────────────

print("\nTraining MLP on XOR...")
print("-" * 45)

model = MLP(learning_rate=0.5)
model.train(X, y, epochs=10000)


# ──────────────────────────────────────────────────────────────
# STEP 4: CHECK PREDICTIONS
# ──────────────────────────────────────────────────────────────

print("\nFinal Predictions:")
print("-" * 40)
print("  A  |  B  | True | Predicted | Prob")
print("-" * 40)
probs = model.forward(X).flatten()
preds = model.predict(X)
for (a, b), true, pred, prob in zip(X, y, preds, probs):
    status = "✓" if true == pred else "✗"
    print(f"  {a}  |  {b}  |  {true}   |     {pred}     | {prob:.4f}  {status}")
print("-" * 40)

acc = np.mean(preds == y) * 100
print(f"\n  Final Accuracy: {acc:.1f}%")


# ──────────────────────────────────────────────────────────────
# STEP 5: PLOT RESULTS
# ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('MLP on XOR Problem', fontsize=13, fontweight='bold')

# --- Plot 1: The XOR data points ---
ax = axes[0]
colors = ['royalblue' if label == 0 else 'tomato' for label in y]
ax.scatter(X[:, 0], X[:, 1], c=colors, s=300, zorder=5, edgecolors='black', linewidths=1.5)
for (a, b), label in zip(X, y):
    ax.annotate(f'({a},{b})→{label}', (a, b), textcoords='offset points',
                xytext=(10, 8), fontsize=10)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_title('XOR Data\n(Blue=0, Red=1)')
ax.set_xlabel('Input A'); ax.set_ylabel('Input B')
ax.grid(True, alpha=0.3)
# Show why one line can't separate them
ax.text(0.5, -0.35, "⚠ No single line can separate Blue from Red!",
        ha='center', fontsize=9, color='gray', style='italic')

# --- Plot 2: Loss curve ---
ax = axes[1]
ax.plot(model.losses, color='steelblue', lw=1.5)
ax.set_title('Training Loss Over Epochs')
ax.set_xlabel('Epoch')
ax.set_ylabel('Binary Cross-Entropy Loss')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')   # log scale helps see the decrease clearly

# --- Plot 3: Decision boundary learned by the MLP ---
ax = axes[2]

# Create a fine grid of points covering the input space
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300),
                     np.linspace(-0.5, 1.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

# Get MLP's probability for every point in the grid
probs_grid = model.forward(grid).reshape(xx.shape)

# Draw the decision boundary (where probability = 0.5)
ax.contourf(xx, yy, probs_grid, levels=50, cmap='RdBu_r', alpha=0.8)
ax.contour(xx, yy, probs_grid, levels=[0.5], colors='black', linewidths=2)

# Overlay the actual XOR points
ax.scatter(X[:, 0], X[:, 1], c=colors, s=300, zorder=5,
           edgecolors='white', linewidths=2)
for (a, b), label in zip(X, y):
    ax.annotate(str(label), (a, b), textcoords='offset points',
                xytext=(0, -18), ha='center', fontsize=11,
                fontweight='bold', color='white')

ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_title('Decision Boundary\n(Black line = 0.5 threshold)')
ax.set_xlabel('Input A'); ax.set_ylabel('Input B')

plt.tight_layout()
plt.savefig('mlp_xor_results.png', dpi=130, bbox_inches='tight')
print("\n  Plot saved as mlp_xor_results.png")
print("\nDone!")

XOR Dataset:
------------------------------
  A  |  B  | Output
------------------------------
  0  |  0  |   0
  0  |  1  |   1
  1  |  0  |   1
  1  |  1  |   0
------------------------------

Training MLP on XOR...
---------------------------------------------
  Epoch  1000 | Loss: 0.054253 | Accuracy: 100.0%
  Epoch  2000 | Loss: 0.003263 | Accuracy: 100.0%
  Epoch  3000 | Loss: 0.001483 | Accuracy: 100.0%
  Epoch  4000 | Loss: 0.000937 | Accuracy: 100.0%
  Epoch  5000 | Loss: 0.000679 | Accuracy: 100.0%
  Epoch  6000 | Loss: 0.000530 | Accuracy: 100.0%
  Epoch  7000 | Loss: 0.000434 | Accuracy: 100.0%
  Epoch  8000 | Loss: 0.000366 | Accuracy: 100.0%
  Epoch  9000 | Loss: 0.000317 | Accuracy: 100.0%
  Epoch 10000 | Loss: 0.000279 | Accuracy: 100.0%

Final Predictions:
----------------------------------------
  A  |  B  | True | Predicted | Prob
----------------------------------------
  0  |  0  |  0   |     0     | 0.0078  ✓
  0  |  1  |  1   |     1     | 0.9841  ✓
  1  |  0  | 